In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t

# ============================================================
# 1. CARGA DE DATOS
# ============================================================

datos = pd.read_csv(
    "../dataset/airfoil_self_noise.dat",
    sep=r"\s+",
    header=None,
    names=[
        "frequency", "angle", "chord_length",
        "velocity", "displacement", "sound_level"
    ]
)

# 1. Datos del modelo: frecuencia y nivel sonoro
x = datos["frequency"].to_numpy()
y = datos["sound_level"].to_numpy()

n = len(y)
x_media = x.mean()
y_media = y.mean()

# 2. Ajustar la recta por mínimos cuadrados
Sxx = np.sum((x - x_media)**2)
Sxy = np.sum((x - x_media) * (y - y_media))

b1 = Sxy / Sxx
b0 = y_media - b1 * x_media

# 3. Estimar la varianza del error
y_estimado = b0 + b1 * x
SCE = np.sum((y - y_estimado)**2)
varianza_estimada = SCE / (n - 2)

# 4. Predicción para 2000 Hz
x_estrella = 2000
y_estrella = b0 + b1 * x_estrella

# 5. Valor crítico t para intervalos bilaterales del 95 %
alpha = 0.05
grados_libertad = n - 2

t_critico = t.ppf(1 - alpha / 2, grados_libertad)

# 6. Factor compartido por ambos errores estándar
factor = (
    1 / n
    + (x_estrella - x_media)**2 / Sxx
)

# 7. Intervalo de confianza para la respuesta media
error_estandar_media = np.sqrt(varianza_estimada * factor)
margen_media = t_critico * error_estandar_media

ic_media_inferior = y_estrella - margen_media
ic_media_superior = y_estrella + margen_media

# 8. Intervalo de predicción para una nueva medición
error_estandar_prediccion = np.sqrt(
    varianza_estimada * (1 + factor)
)
margen_prediccion = t_critico * error_estandar_prediccion

ip_inferior = y_estrella - margen_prediccion
ip_superior = y_estrella + margen_prediccion

# 9. Mostrar resultados
print(f"Frecuencia elegida: {x_estrella} Hz")
print(f"Nivel sonoro predicho: {y_estrella:.6f} dB")
print(f"Grados de libertad: {grados_libertad}")
print(f"t crítico: {t_critico:.6f}")

print("\nINTERVALO PARA LA RESPUESTA MEDIA")
print(f"Error estándar: {error_estandar_media:.6f} dB")
print(f"Margen: ±{margen_media:.6f} dB")
print(
    f"IC del 95 %: "
    f"[{ic_media_inferior:.6f}, {ic_media_superior:.6f}] dB"
)

print("\nINTERVALO PARA UNA NUEVA MEDICIÓN")
print(f"Error estándar: {error_estandar_prediccion:.6f} dB")
print(f"Margen: ±{margen_prediccion:.6f} dB")
print(
    f"IP del 95 %: "
    f"[{ip_inferior:.6f}, {ip_superior:.6f}] dB"
)



Frecuencia elegida: 2000 Hz
Nivel sonoro predicho: 125.593780 dB
Grados de libertad: 1501
t crítico: 1.961546

INTERVALO PARA LA RESPUESTA MEDIA
Error estándar: 0.170212 dB
Margen: ±0.333880 dB
IC del 95 %: [125.259900, 125.927659] dB

INTERVALO PARA UNA NUEVA MEDICIÓN
Error estándar: 6.354700 dB
Margen: ±12.465034 dB
IP del 95 %: [113.128746, 138.058813] dB
